# Binary Classification with mlcore

This notebook demonstrates the complete workflow for binary classification using mlcore.

Topics covered:

- Registry exploration
- Model training
- Prediction generation
- Probability estimation
- Classification metrics
- Model metadata inspection

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import mlcore.classification

from mlcore.core.registry import MODEL_REGISTRY
from mlcore.core.trainer import Trainer

from mlcore.evaluation.classification import (
    evaluate_classification,
)

In [2]:
MODEL_REGISTRY.list()

['adaboost',
 'bagging',
 'bernoulli_nb',
 'categorical_nb',
 'complement_nb',
 'decision_tree',
 'extra_tree',
 'extra_trees',
 'gaussian_nb',
 'gaussian_process',
 'gradient_boosting',
 'hist_gradient_boosting',
 'knn',
 'lda',
 'lgbm_classifier',
 'linear_svc',
 'logistic_regression',
 'multinomial_nb',
 'nearest_centroid',
 'nu_svc',
 'qda',
 'radius_neighbors',
 'random_forest',
 'ridge_classifier',
 'sgd_classifier',
 'svc',
 'xgb_classifier',
 'xgb_rf_classifier']

In [3]:
X, y = make_classification(
    n_samples=1000,
    n_features=50,
    n_informative=20,
    n_redundant=10,
    n_classes=2,
    random_state=42,
)

print(X.shape)
print(y.shape)

(1000, 50)
(1000,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(X_train.shape)
print(X_test.shape)

(800, 50)
(200, 50)


In [5]:
trainer = Trainer(
    registry=MODEL_REGISTRY,
)

In [6]:
result = trainer.fit(
    algorithm="random_forest",
    X=X_train,
    y=y_train,
    n_estimators=100,
    max_depth=10,
    random_state=42,
    return_predictions=True,
    return_probabilities=True,
)

In [7]:
result

TrainResult(model=RandomForestClassifier(max_depth=10, random_state=42), backend=ClassificationBackend(status=trained, n_metrics=0, n_metadata=2), spec=AlgorithmSpec(backend='sklearn', task='classification', name='random_forest', runner=<function make_classifier_runner.<locals>.runner at 0x7fccea921bc0>, backend_cls=<class 'mlcore.classification.backend.ClassificationBackend'>, aliases=(), default_params={}, search_space={}, tags=('classification', 'sklearn'), supports_cv=False, supports_proba=True, description=None), metrics={}, predictions=array([1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1,
       0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 1

In [8]:
backend = result.backend

backend

ClassificationBackend(status=trained, n_metrics=0, n_metadata=2)

In [9]:
backend.get_metadata()

{'classes': array([0, 1]), 'n_features': 50}

In [10]:
predictions = trainer.predict(
    result,
    X_test,
)

predictions[:10]

array([1, 0, 0, 0, 1, 1, 1, 1, 0, 0])

In [11]:
probabilities = trainer.predict_proba(
    result,
    X_test,
)

probabilities[:5]

array([[0.2428812 , 0.7571188 ],
       [0.57716362, 0.42283638],
       [0.85881817, 0.14118183],
       [0.71768629, 0.28231371],
       [0.33231158, 0.66768842]])

In [12]:
metrics = evaluate_classification(
    y_true=y_test,
    y_pred=predictions,
)

metrics

{'accuracy': 0.88,
 'balanced_accuracy': 0.88,
 'precision': 0.9130434782608695,
 'recall': 0.84,
 'sensitivity': 0.84,
 'specificity': 0.92,
 'f1': 0.875,
 'mcc': 0.7624437362098715}

In [13]:
pd.DataFrame(
    metrics,
    index=["Random Forest"],
).T

,Random Forest
accuracy,0.880000
balanced_accuracy,0.880000
precision,0.913043
recall,0.840000
sensitivity,0.840000
specificity,0.920000
f1,0.875000
mcc,0.762444


In [14]:
spec = MODEL_REGISTRY.get(
    "random_forest",
)

spec

AlgorithmSpec(backend='sklearn', task='classification', name='random_forest', runner=<function make_classifier_runner.<locals>.runner at 0x7fccea921bc0>, backend_cls=<class 'mlcore.classification.backend.ClassificationBackend'>, aliases=(), default_params={}, search_space={}, tags=('classification', 'sklearn'), supports_cv=False, supports_proba=True, description=None)

In [15]:
print("Name:", spec.name)
print("Backend:", spec.backend)
print("Task:", spec.task)
print("Tags:", spec.tags)

Name: random_forest
Backend: sklearn
Task: classification
Tags: ('classification', 'sklearn')


## Summary

In this notebook we:

- Loaded a binary classification dataset.
- Trained a Random Forest classifier.
- Generated predictions and probabilities.
- Evaluated model performance.
- Explored model metadata.
- Inspected the registry and algorithm specification.

This workflow represents the basic training cycle in mlcore.